In [1]:
// In python use: from pyspark.sql.functions import broadcast, split, lit
import org.apache.spark.sql.functions.{broadcast, split, lit}

Intitializing Scala interpreter ...

Spark Web UI available at http://2cdd78297c7e:4042
SparkContext available as 'sc' (version = 3.5.1, master = local[*], app id = local-1733318131370)
SparkSession available as 'spark'


import org.apache.spark.sql.functions.{broadcast, split, lit}


In [1]:
val matchesBucketed = spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/matches.csv")

Intitializing Scala interpreter ...

Spark Web UI available at http://2cdd78297c7e:4042
SparkContext available as 'sc' (version = 3.5.1, master = local[*], app id = local-1733598142195)
SparkSession available as 'spark'


matchesBucketed: org.apache.spark.sql.DataFrame = [match_id: string, mapid: string ... 8 more fields]


In [2]:
val matchDetailsBucketed =  spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/match_details.csv")

matchDetailsBucketed: org.apache.spark.sql.DataFrame = [match_id: string, player_gamertag: string ... 34 more fields]


In [3]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.matches_bucketed""")
val bucketedDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
   match_id STRING,
   is_team_game BOOLEAN,
   playlist_id STRING,
   completion_date TIMESTAMP
)
USING iceberg
PARTITIONED BY (completion_date, bucket(4, match_id));
"""
spark.sql(bucketedDDL)

bucketedDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
   match_id STRING,
   is_team_game BOOLEAN,
   playlist_id STRING,
   completion_date TIMESTAMP
)
USING iceberg
PARTITIONED BY (completion_date, bucket(4, match_id));
"
res0: org.apache.spark.sql.DataFrame = []


In [4]:
matchesBucketed.select(
   $"match_id", $"is_team_game", $"playlist_id", $"completion_date"
   )
   .write.mode("append")
   .partitionBy("completion_date")
 .bucketBy(4, "match_id").format("parquet").saveAsTable("bootcamp.matches_bucketed")

In [5]:
val bucketedDetailsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
   match_id STRING,
   player_gamertag STRING,
   player_total_kills INTEGER,
   player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(8, match_id));
"""
spark.sql(bucketedDetailsDDL)

matchDetailsBucketed.select(
   $"match_id", $"player_gamertag", $"player_total_kills", $"player_total_deaths")
   .write.mode("append")
 .bucketBy(8, "match_id").saveAsTable("bootcamp.match_details_bucketed")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


bucketedDetailsDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
   match_id STRING,
   player_gamertag STRING,
   player_total_kills INTEGER,
   player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(8, match_id));
"


In [8]:
matchesBucketed.createOrReplaceTempView("matches")
matchDetailsBucketed.createOrReplaceTempView("match_details")

spark.sql("""
     SELECT * FROM bootcamp.match_details_bucketed mdb 
     JOIN bootcamp.matches_bucketed md 
     ON mdb.match_id = md.match_id
     AND md.completion_date = DATE('2016-01-01')        
""").explain()


spark.sql("""
    SELECT * FROM match_details mdb 
    JOIN matches md ON mdb.match_id = md.match_id        
""").explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#170], [match_id#174], Inner
   :- Sort [match_id#170 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#170, 200), ENSURE_REQUIREMENTS, [plan_id=104]
   :     +- BatchScan demo.bootcamp.match_details_bucketed[match_id#170, player_gamertag#171, player_total_kills#172, player_total_deaths#173] demo.bootcamp.match_details_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []
   +- Sort [match_id#174 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(match_id#174, 200), ENSURE_REQUIREMENTS, [plan_id=105]
         +- BatchScan demo.bootcamp.matches_bucketed[match_id#174, is_team_game#175, playlist_id#176, completion_date#177] demo.bootcamp.matches_bucketed (branch=null) [filters=completion_date IS NOT NULL, completion_date = 1451606400000000, match_id IS NOT NULL, groupedBy=] RuntimeFilters: []


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=fa

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1000000000000")

val broadcastFromThreshold = matches.as("m").join(matchDetails.as("md"), $"m.match_id" === $"md.match_id")
  .select($"m.completion_date", $"md.player_gamertag",  $"md.player_total_kills")
  .take(5)

val explicitBroadcast = matches.as("m").join(broadcast(matchDetails).as("md"), $"m.match_id" === $"md.match_id")
  .select($"md.*", split($"completion_date", " ").getItem(0).as("ds"))

val bucketedValues = matchDetailsBucketed.as("mdb").join(matchesBucketed.as("mb"), $"mb.match_id" === $"mdb.match_id").explain()
// .take(5)

val values = matchDetailsBucketed.as("m").join(matchesBucketed.as("md"), $"m.match_id" === $"md.match_id").explain()

explicitBroadcast.write.mode("overwrite").insertInto("match_details_bucketed")

matches.withColumn("ds", split($"completion_date", " ").getItem(0)).write.mode("overwrite").insertInto("matches_bucketed")

spark.sql(bucketedSQL)

<console>: 27: error: not found: value matches